In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)

In [ ]:
de=pd.read_csv("/g/korbel/olisov/Visium_Linda/stiffness/DE/11samples/825_zinbw_edgeR_all_final.tsv", sep='\t')
de['names']=de.index
#de=de[de['FDR'] > 7.154940e-10]

In [ ]:
de

In [ ]:
# Prepare data
de['-log10(pval)'] = -np.log10(de['FDR'])
de['significance'] = 'Not Significant'
de.loc[(de['FDR'] < 0.05) & (de['logFC'] >= 0.5), 'significance'] = 'Upregulated'
de.loc[(de['FDR'] < 0.05) & (de['logFC'] <= -0.5), 'significance'] = 'Downregulated'

# Top significant upregulated/downregulated genes (by FDR)
top_up = de.query("significance == 'Upregulated'").sort_values(by='FDR').head(10)
top_down = de.sort_values(by='FDR', ascending=True).head(10)
top_genes = pd.concat([top_up, top_down])

# Set figure size
fig_width_mm = 90  # wider to fit legend
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.75
fig, ax = plt.subplots(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Volcano plot
sns.scatterplot(
    data=de,
    x='logFC',
    y='-log10(pval)',
    hue='significance',
    palette={
        'Not Significant': 'gray',
        'Upregulated': 'red',
        'Downregulated': 'blue'},
    alpha=0.7,
    ax=ax, s=4)

ax.set_xlim(-2.3, 2.1)
# Axes labels
ax.set_xlabel('Log Fold Change', fontsize=5, labelpad=2)
ax.set_ylabel('-log₁₀P-value', fontsize=5, labelpad=2)

# Ticks
ax.tick_params(axis='both', which='major', labelsize=5, width=0.3, pad=1)

# Remove top/right spines
sns.despine(ax=ax, top=True, right=True)

# Annotate top genes
for _, row in top_genes.iterrows():
    ax.text(
        row['logFC'] - 0.01,
        row['-log10(pval)'] + 0.04,
        row['names'],
        fontsize=3,
        ha='right',
        va='center')

# Adjust legend
legend = ax.legend(
    title_fontsize=5,
    fontsize=5,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=False
)

# Final layout
plt.tight_layout()
plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/11samples/volcano_plot_825.svg', bbox_inches='tight', dpi=600)
plt.show()

# Venn

In [ ]:
#compare 240, 480 and 720 stiffness values
de1=pd.read_csv("/g/korbel/olisov/Visium_Linda/stiffness/DE/11samples/275_zinbw_edgeR_all_final.tsv", sep='\t')
de2=pd.read_csv("/g/korbel/olisov/Visium_Linda/stiffness/DE/11samples/550_zinbw_edgeR_all_final.tsv", sep='\t')
de3=pd.read_csv("/g/korbel/olisov/Visium_Linda/stiffness/DE/11samples/825_zinbw_edgeR_all_final.tsv", sep='\t')
rf=pd.read_csv('/g/korbel/olisov/Visium_Linda/stiffness/regression/final_RF_11samples_topgenes_um50.tsv', sep='\t')
de1['names']=de1.index
de1=de1[(de1['logFC']>=0.5)&(de1['FDR']<=0.05)]
de2['names']=de2.index
de2=de2[(de2['logFC']>=0.5)&(de2['FDR']<=0.05)]
de3['names']=de3.index
de3=de3[(de3['logFC']>=0.5)&(de3['FDR']<=0.05)]
rf=rf.head(34) #top 5% of features

In [ ]:
# --- Extract gene sets ---
set1 = set(de1['names'].astype(str))
set2 = set(de2['names'].astype(str))
set3 = set(de3['names'].astype(str))
set4 = set(rf['Feature'].astype(str))

# --- Compute intersection ---
overlap12=sorted(set1&set2)
overlap = sorted(set1 & set2 & set3 & set4)

In [ ]:
overlap

In [ ]:
from venny4py.venny4py import *

okabe_ito = [
    '#E69F00',  # orange
    '#5AB4E5', #blue
    '#009E73',   # green
    '#CC79A7',  # purple
]

# Assuming you have the four dataframes in the 'tops' dictionary
sets = {
    '275 Pa': set1,
    '550 Pa': set2,
    '825 Pa': set3,
    'RF': set4}

# Create the Venn diagram with venny4py
venny4py(sets=sets, legend_cols=4, colors=okabe_ito)
out = "/g/korbel/olisov/Visium_Linda/paper_figures/11samples/venn_DE_RF.svg"
plt.savefig(out, dpi=300, bbox_inches='tight')